# Database-Driven AcroForm Fill — End-to-End

This notebook walks through filling an editable (AcroForm) PDF from a canonical database JSON, end-to-end, on a single input PDF. Every intermediate artifact is written to `output/` so the steps are inspectable.

**Python:** tested on **3.12**, requires **>= 3.10**.

**Standalone behavior:** every algorithmic step is inlined so the reader can see the actual implementation. The only `claims_parser` imports are pydantic data models (so the JSON artifacts stay typed) and the AcroForm writer (`write_acroform` — ~200 lines of pymupdf widget-update logic that's not algorithmically interesting for a demo).

Pipeline:

1. **Detect** the PDF carries AcroForm widgets.
2. **Extract** widgets → `FormSchema` + `AcroFormBinding` (pymupdf).
3. **Load** the canonical DB JSON (envelope strip + `DBEnvelope` validation).
    - **3a.** (Optional) **LLM-fill** empty fields in the DB template.
4. **Canonical DB paths** + DB-schema fingerprint (form-independent).
5. **Schema fingerprint** (form-specific cache key).
6. **Heuristic resolve** — no-LLM fast-path (token Jaccard with section bias).
7. **LLM resolve** — structured-output batch for residuals.
8. **Persist** the `FieldMap` (cache, form-keyed).
9. **Materialize** — apply `Resolution`s + DB → `FilledFormSchema` (deterministic).
10. **Write** values into AcroForm widgets.
11. **Summary**.

Change `INPUT_PDF` and `DB_TEMPLATE` in the setup cell to demo a different form or case.

## 0. Setup

All paths are relative to the repo root. The cell below walks up the filesystem until it finds `claims_parser/`, then `chdir`s there so cached artifacts under `output/intermediate/` resolve correctly. Secrets (OpenAI key) live in `parser.env`.

In [ ]:
import os, sys
from pathlib import Path

assert sys.version_info >= (3, 10), "This notebook requires Python 3.10+"

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "claims_parser").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)
print("Python   :", sys.version.split()[0])

# --- inputs ---
INPUT_PDF    = Path("input/AETNA_form2.pdf")            # change me
DB_TEMPLATE  = Path("database/database_schema.json")    # change me

# --- outputs (all derived from the PDF stem) ---
OUT_DIR = Path("output/intermediate")
OUT_DIR.mkdir(parents=True, exist_ok=True)
Path("output/final").mkdir(parents=True, exist_ok=True)
STEM = INPUT_PDF.stem

SCHEMA_JSON    = OUT_DIR / f"{STEM}.schema.json"
BINDING_JSON   = OUT_DIR / f"{STEM}.binding.json"
DB_FILLED_JSON = Path("database") / f"{INPUT_PDF.stem}.case.json"
FILLMAP_JSON   = OUT_DIR / f"{STEM}.fillmap.json"
FILLED_JSON    = OUT_DIR / f"{STEM}.filled.json"
FILLED_PDF     = Path("output/final") / f"{STEM}.filled.pdf"
REVIEW_JSON    = OUT_DIR / f"{STEM}.review.json"

print("Input PDF :", INPUT_PDF)
print("DB template:", DB_TEMPLATE)

### Tech stack used by this notebook

- **pymupdf** — widget enumeration + AcroForm field updates.
- **openai** — `client.chat.completions.parse` with `response_format=PydanticModel` for structured outputs (gpt-5-mini).
- **pydantic v2** — every JSON artifact is typed; LLM responses validate against the same models.
- **python-dotenv** — load `OPENAI_API_KEY` from `parser.env`.

In [ ]:
import json, hashlib, re
from datetime import datetime, timezone
from typing import Any, Optional

import pymupdf
import dotenv
from openai import OpenAI

# Data models only — no helpers from claims_parser.
from claims_parser.schema_models import FieldType, FormField, FormSchema
from claims_parser.filler_models import FilledFormField, FilledFormSchema
from claims_parser.acroform_models import (
    AcroFormBinding, AcroFormFieldBinding, OptionBinding,
)
from claims_parser.db_template_models import DBEnvelope
from claims_parser.db_mapping_models import (
    Resolution, OptionResolution, FieldResolution, FieldMap,
    DerivedSpec, LLMResolverResponse,
)

# The AcroForm writer is imported (not inlined): ~200 lines of pymupdf
# widget-update logic, not algorithmically interesting for this demo.
from claims_parser.acroform_writer import write_acroform
from claims_parser.review_models import ReviewReport

dotenv.load_dotenv(REPO_ROOT / "parser.env")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY missing — add it to parser.env"

print("pymupdf  :", pymupdf.__version__)
import importlib.metadata as _md
print("openai   :", _md.version("openai"))
print("pydantic :", _md.version("pydantic"))
print("dotenv   :", _md.version("python-dotenv"))

## 1. Detect — is this PDF editable?

Open with pymupdf and check whether any page has AcroForm widgets. Only AcroForm PDFs flow through this notebook; scanned forms use the non-editable branch (Azure DI + anchor placer + vision passes — out of scope here).

In [ ]:
def is_acroform_pdf(pdf_path: Path) -> bool:
    doc = pymupdf.open(pdf_path)
    try:
        for page in doc:
            if any(True for _ in page.widgets()):
                return True
        return False
    finally:
        doc.close()

print("is_acroform_pdf:", is_acroform_pdf(INPUT_PDF))
print("branch         :", "acroform" if is_acroform_pdf(INPUT_PDF) else "scanned")

## 2. Extract widgets → `FormSchema` + `AcroFormBinding`

Walk every page, enumerate every widget, infer a label (PDF `TU` tooltip → `field_label` → `field_name` fallback), group radio buttons by their parent `field_name`, and emit:

- a **`FormSchema`** — generic, shape-compatible with the non-editable pipeline.
- an **`AcroFormBinding`** — ties each `field_id` to widget xref(s) so the writer knows which widget to update.

Production code (`claims_parser.acroform_extractor`) additionally runs a Tier-2 text-layer proximity search and a Tier-3 vision LLM pass for widgets without TU tooltips, and an LLM type-refinement pass. Those are skipped here for clarity — the AETNA forms have decent tooltips so TU alone yields a clean schema. Pass `--no-refine` / `--no-vision` to the CLI for the same effect.

In [ ]:
LONG_TEXT_MIN_HEIGHT_PT = 30.0

def _snake(label: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9]+", "_", label.strip().lower()).strip("_")
    return s or "field"

def _uniqify(field_id: str, taken: set[str]) -> str:
    if field_id not in taken:
        taken.add(field_id); return field_id
    i = 2
    while f"{field_id}_{i}" in taken: i += 1
    out = f"{field_id}_{i}"
    taken.add(out); return out

def _category_for_widget(widget) -> str:
    t = (widget.field_type_string or "").lower()
    if t == "checkbox":    return "checkbox"
    if t == "radiobutton": return "radiobutton"
    if t in ("listbox", "combobox"): return "choice"
    if t == "signature":   return "signature"
    return "text"

def _on_value(widget) -> str:
    try: return widget.on_state() or "Yes"
    except Exception: return "Yes"

def extract_acroform(pdf_path: Path) -> tuple[FormSchema, AcroFormBinding]:
    doc = pymupdf.open(pdf_path)
    try:
        # Pass 1: collect every widget with TU/field_label
        records: list[dict] = []
        for page_num, page in enumerate(doc, start=1):
            for w in page.widgets() or []:
                label = (getattr(w, "field_label", None) or w.field_name or "").strip()
                r = w.rect
                records.append(dict(
                    widget=w, page=page_num, category=_category_for_widget(w),
                    label=label, xref=w.xref, field_name=w.field_name or "",
                    rect=(r.x0, r.y0, r.x1, r.y1),
                ))

        # Pass 2: group radio buttons by parent field_name
        radio_groups: dict[str, list[dict]] = {}
        non_radio: list[dict] = []
        for rec in records:
            if rec["category"] == "radiobutton":
                radio_groups.setdefault(rec["field_name"], []).append(rec)
            else:
                non_radio.append(rec)

        # Pass 3: build draft FormFields + AcroFormFieldBindings
        taken: set[str] = set()
        fields: list[FormField] = []
        bindings: list[AcroFormFieldBinding] = []

        for rec in non_radio:
            label = rec["label"] or rec["field_name"] or "field"
            fid = _uniqify(_snake(label), taken)
            cat = rec["category"]
            w = rec["widget"]
            options: Optional[list[str]] = None
            on_val: Optional[str] = None

            if cat == "checkbox":
                ftype: FieldType = "checkbox"
                options = [label]
                on_val = _on_value(w)
                bind_cat = "checkbox"
            elif cat == "choice":
                ftype = "radio_group"
                cv = list(getattr(w, "choice_values", None) or [])
                options = cv if cv else None
                bind_cat = "choice"
            elif cat == "signature":
                ftype = "signature"
                bind_cat = "signature"
            else:
                ftype = "long_text" if (rec["rect"][3] - rec["rect"][1]) > LONG_TEXT_MIN_HEIGHT_PT else "text"
                bind_cat = "text"

            fields.append(FormField(
                field_id=fid, label=label, section=None, field_type=ftype,
                options=options, format_hint=None, required=False, source_text=label,
            ))
            bindings.append(AcroFormFieldBinding(
                field_id=fid, widget_category=bind_cat,
                page=rec["page"], widget_xref=rec["xref"],
                widget_name=rec["field_name"] or None,
                rect=list(rec["rect"]), on_value=on_val,
            ))

        # Radio groups: one FormField per parent
        for parent_name, members in radio_groups.items():
            group_label = parent_name or "choice"
            fid = _uniqify(_snake(group_label), taken)
            option_labels: list[str] = []
            option_bindings: list[OptionBinding] = []
            for m in members:
                lbl = m["label"] or _on_value(m["widget"])
                option_labels.append(lbl)
                option_bindings.append(OptionBinding(
                    label=lbl, widget_xref=m["xref"], widget_name=m["field_name"],
                    on_value=_on_value(m["widget"]), page=m["page"],
                ))
            fields.append(FormField(
                field_id=fid, label=group_label, section=None, field_type="radio_group",
                options=option_labels, format_hint=None, required=False, source_text=group_label,
            ))
            bindings.append(AcroFormFieldBinding(
                field_id=fid, widget_category="radio_group",
                page=members[0]["page"], options=option_bindings,
            ))

        return (
            FormSchema(form_title=None, issuer=None, sections=[], fields=fields),
            AcroFormBinding(file_name=pdf_path.name, fields=bindings),
        )
    finally:
        doc.close()

schema, binding = extract_acroform(INPUT_PDF)
SCHEMA_JSON.write_text(schema.model_dump_json(indent=2))
BINDING_JSON.write_text(binding.model_dump_json(indent=2))
print(f"Schema  -> {SCHEMA_JSON}")
print(f"Binding -> {BINDING_JSON}")
print(f"  {len(schema.fields)} fields, {len(binding.fields)} bindings")
by_type: dict[str, int] = {}
for f in schema.fields:
    by_type[f.field_type] = by_type.get(f.field_type, 0) + 1
for k, v in sorted(by_type.items()):
    print(f"    {k:11s} {v}")

## 3. Load the canonical DB JSON

The DB JSON is the **single source of truth** for every form. Its shape is **fixed** across all cases (validated against `DBEnvelope`); only the values and `claim.serviceLines` cardinality vary case-to-case.

Below: strip the envelope (`{success, message, result, executionTimeSec}` → the inner `result`), validate, and show a few top-level keys.

In [ ]:
def load_db_json(path: Path) -> dict:
    raw = json.loads(path.read_text())
    if not isinstance(raw, dict) or "result" not in raw:
        raise ValueError(f"{path}: expected an object with a top-level 'result' key")
    result = raw["result"]
    if not isinstance(result, dict):
        raise ValueError(f"{path}: 'result' must be an object")
    return result

db = load_db_json(DB_TEMPLATE)
print(f"loaded {DB_TEMPLATE} -> result keys: {list(db.keys())}")
print(f"patient.firstName   = {db['patient']['firstName']!r}")
print(f"claim.claimNumber   = {db['claim']['claimNumber']!r}")
print(f"serviceLines count  = {len(db['claim']['serviceLines'])}")

# Validate against the canonical pydantic model so structural drift fails loudly here
env_obj = DBEnvelope.model_validate(json.loads(DB_TEMPLATE.read_text()))
print("DBEnvelope validation: OK")

## 3a. (Optional) LLM-fill empty fields in the DB template

If the input template has empty strings/lists/`null`s (typical in dev/testing), we can fill them once with a single LLM call against the **fixed** `DBEnvelope` shape. The same canonical patient/case then drives every form.

In production this step is skipped — the real DB JSON arrives populated.

The system prompt below preserves any existing non-empty values and forbids adding new keys. The response validates against `DBEnvelope` so the model cannot deviate from the canonical schema.

In [ ]:
SYSTEM_PROMPT_TEMPLATE_FILL = """You receive a JSON document representing a healthcare claim
appeal record. Some fields are populated; others are empty strings, empty lists,
or null. Return the SAME document with every empty field filled with realistic,
internally-consistent fictional values.

Rules:
- Do not change any field that already has a non-empty value.
- Keep all field names exactly as given (including any typos in source keys).
- Values must be consistent: patient name, DOB, addresses, IDs all belong to
  the same fictional individual; dates are mutually plausible; amounts add up.
- Date format: keep the format consistent with whatever other dates use.
  If all dates are empty, use ISO YYYY-MM-DD.
- Service lines: if the array has entries already, fill in their empty fields.
  If the array is empty, add 2-4 plausible service lines.
- Never use real personal data.
- Do not invent NEW top-level keys or sub-objects. Only fill what the schema
  defines. The response must validate against the provided pydantic schema.
"""

def fill_db_template(template: dict, model: str = "gpt-5-mini") -> dict:
    client = OpenAI(api_key=OPENAI_API_KEY)
    normalized = DBEnvelope.model_validate(template).model_dump()
    user_msg = json.dumps(normalized, indent=2)
    resp = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_TEMPLATE_FILL},
            {"role": "user",   "content": user_msg},
        ],
        response_format=DBEnvelope,
    )
    parsed = resp.choices[0].message.parsed
    if parsed is None:
        raise RuntimeError(f"empty parse result; finish_reason={resp.choices[0].finish_reason}")
    return parsed.model_dump()

# Detect whether the template has empties that need filling
raw_envelope = json.loads(DB_TEMPLATE.read_text())
_flat_sample = json.dumps(raw_envelope)
needs_fill = ('"":' in _flat_sample) or ('""' in _flat_sample)

if needs_fill:
    print("Template has empty fields. Running LLM template-fill...")
    filled_envelope = fill_db_template(raw_envelope)
    DB_FILLED_JSON.parent.mkdir(parents=True, exist_ok=True)
    DB_FILLED_JSON.write_text(json.dumps(filled_envelope, indent=2))
    db = filled_envelope["result"]
    print(f"  wrote {DB_FILLED_JSON}")
    print(f"  provider.renderingProviderName = {db['provider']['renderingProviderName']!r}")
else:
    print("Template appears already populated; skipping LLM fill.")
    print(f"Using DB at: {DB_TEMPLATE}")

## 4. Canonical DB paths + DB-schema fingerprint

The DB schema is **fixed**, so the set of available leaf paths is too. Deriving them from `DBEnvelope` (pydantic) rather than from any particular instance keeps the fingerprint stable even when `claim.serviceLines` happens to be empty in one case and populated in another.

List indices are normalized to `[]` so `claim.serviceLines[0].procedureCode` and `claim.serviceLines[1].procedureCode` collapse to a single canonical path `claim.serviceLines[].procedureCode`.

In [ ]:
import typing as _t

def canonical_db_paths() -> list[str]:
    def walk(model_cls, prefix: str, out: list[str]) -> None:
        try:
            fields = model_cls.model_fields
        except AttributeError:
            out.append(prefix); return
        for name, info in fields.items():
            key = f"{prefix}.{name}" if prefix else name
            ann = info.annotation
            origin = _t.get_origin(ann)
            args = _t.get_args(ann)
            if origin is list:
                inner = args[0] if args else None
                if inner is not None and hasattr(inner, "model_fields"):
                    walk(inner, key + "[]", out)
                else:
                    out.append(key + "[]")
            elif hasattr(ann, "model_fields"):
                walk(ann, key, out)
            else:
                out.append(key)
    paths: list[str] = []
    walk(DBEnvelope, "", paths)
    # The rest of the pipeline treats envelope-stripped `result` as the root.
    return sorted(set(
        p[len("result."):] for p in paths if p.startswith("result.")
    ))

def db_schema_fingerprint() -> str:
    body = "\n".join(canonical_db_paths())
    return hashlib.sha256(body.encode()).hexdigest()[:16]

ALL_PATHS = canonical_db_paths()
print(f"canonical paths: {len(ALL_PATHS)}")
for p in ALL_PATHS[:12]:
    print(f"  {p}")
print("  ...")
DB_FP = db_schema_fingerprint()
print(f"\nDB schema fingerprint: {DB_FP}")

## 5. Schema fingerprint (form-specific cache key)

Hash the **shape** of the `FormSchema` (field_id, label, section, field_type, sorted options) — not `source_text`, which is spatial. Together `(schema_fp, db_schema_fp)` is the cache key for the resolved `FieldMap`. Same form filled for 500 patients = 1 LLM resolution.

In [ ]:
def compute_schema_fingerprint(schema: FormSchema) -> str:
    body = json.dumps(
        [
            dict(
                field_id=f.field_id, label=f.label, section=f.section,
                field_type=f.field_type,
                options=sorted(f.options) if f.options else None,
            )
            for f in schema.fields
        ],
        sort_keys=True,
    )
    return hashlib.sha256(body.encode()).hexdigest()[:16]

SCHEMA_FP = compute_schema_fingerprint(schema)
print(f"schema_fp:           {SCHEMA_FP}")
print(f"db_schema_fp:        {DB_FP}")
print(f"fillmap cache file:  {FILLMAP_JSON}")

## 6. Heuristic resolve — no-LLM fast-path

For each `FormField`, compute the token-set Jaccard similarity between the field's identifiers (`field_id`, `label`, `section`) and every canonical DB path. Split camelCase so `memberID` matches `member_id`. Boost paths under the block hinted by the field's section (`patient`, `subscriber`, `payer`, etc.). Filter by `field_type` so `field_type=date` only considers date-shaped paths.

If similarity ≥ 0.7, emit a confident `Resolution(kind="scalar")`. Choice fields (`radio_group`/`checkbox`) always punt to the LLM — option mapping is too nuanced for token overlap.

In [ ]:
_CAMEL_SPLIT = re.compile(r"(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])")

_DATE_PATH_HINTS  = ("date", "dob", "dateofbirth", "dateofservice",
                     "effectivedate", "datefrom", "dateto", "datereceived",
                     "submissiondate", "filingdate")
_PHONE_PATH_HINTS = ("phone", "fax")
_EMAIL_PATH_HINTS = ("email",)
_BLOCK_HINTS = {
    "patient":    ("patient",),
    "subscriber": ("subscriber", "insured", "policyholder"),
    "dependent":  ("dependent",),
    "provider":   ("provider", "facility", "billing", "rendering", "servicing"),
    "payer":      ("payer", "insurer", "insurance", "plan"),
    "claim":      ("claim", "appeal", "denial"),
}

def _tokens(s: str) -> set[str]:
    """Split snake_case, camelCase, and mixed into a lowercase token set."""
    decamel = _CAMEL_SPLIT.sub(" ", s)
    return set(re.findall(r"[a-z0-9]+", decamel.lower()))

def _block_for_section(section: Optional[str]) -> Optional[str]:
    if not section: return None
    sl = section.lower()
    for block, hints in _BLOCK_HINTS.items():
        if any(h in sl for h in hints):
            return block
    return None

def _default_transform(field: FormField) -> Optional[str]:
    if field.field_type == "date":    return f"date:{field.format_hint or 'YYYY-MM-DD'}"
    if field.field_type == "phone":   return "phone:formatted"
    if field.field_type == "currency": return "currency"
    if field.field_type == "number":  return "number"
    return None

def heuristic_resolve(field: FormField, paths: list[str], floor: float = 0.7) -> Optional[FieldResolution]:
    if field.field_type in ("checkbox", "radio_group"):
        return None  # option mapping is too nuanced for token overlap
    f_toks = _tokens(field.field_id) | _tokens(field.label)
    if field.section: f_toks |= _tokens(field.section)
    if not f_toks: return None
    preferred_block = _block_for_section(field.section)

    best_score, best_path = 0.0, ""
    for path in paths:
        p_toks = _tokens(path)
        if not p_toks: continue
        if field.field_type == "date"  and not any(h in path.lower() for h in _DATE_PATH_HINTS):  continue
        if field.field_type == "phone" and not any(h in path.lower() for h in _PHONE_PATH_HINTS): continue
        if field.field_type == "email" and not any(h in path.lower() for h in _EMAIL_PATH_HINTS): continue
        jac = len(f_toks & p_toks) / max(1, len(f_toks | p_toks))
        if preferred_block and path.startswith(preferred_block + "."):
            jac += 0.15
        if jac > best_score:
            best_score, best_path = jac, path

    if best_score < floor or not best_path:
        return None
    return FieldResolution(
        field_id=field.field_id, confidence="high",
        value=Resolution(
            kind="scalar", paths=[best_path],
            transform=_default_transform(field),
            reason=f"heuristic jaccard={best_score:.2f}",
        ),
    )

heuristic_hits: dict[str, FieldResolution] = {}
residuals: list[FormField] = []
for f in schema.fields:
    hit = heuristic_resolve(f, ALL_PATHS)
    if hit is not None:
        heuristic_hits[f.field_id] = hit
    else:
        residuals.append(f)

print(f"heuristic resolved: {len(heuristic_hits)}/{len(schema.fields)}")
for fid, fr in list(heuristic_hits.items())[:8]:
    print(f"  {fid:35s} -> {fr.value.paths[0]:50s} ({fr.value.reason})")
print(f"\nLLM residuals    : {len(residuals)}")

## 7. LLM resolve — structured-output batch for residuals

Send every unresolved field, the canonical path list, and a small set of sample DB values to `gpt-5-mini` in a single call. The response validates against `LLMResolverResponse` (pydantic), which holds a `list[FieldResolution]`. The model picks `kind` per field: `scalar`, `compose` (with template + multiple paths), `service_line` (table row), `derived` (predicate), `constant`, or `unmapped`. Choice fields get one `OptionResolution` per option.

The system prompt constrains the model: never invent paths outside the given list; never reference form-specific labels/options in reasoning.

In [ ]:
LLM_SYSTEM_PROMPT = """You receive a list of form fields and the FLATTENED list
of available DB paths (dotted notation; list indices are normalized to "[]").

For each form field, produce a FieldResolution. Allowed `value.kind` values:
- "scalar"       : single DB path supplies the value.
- "compose"      : combine multiple paths via a Python str.format template
                   such as "{0} {1} {2}" (first/middle/last name, for example).
- "service_line" : the field belongs to a table row; use one path with the
                   literal "[{i}]" placeholder where the row index goes, and
                   set `service_line_index` to the 0-based row index inferred
                   from the field_id or label.
- "derived"      : the value is a small predicate over the DB (eq, neq,
                   non_empty, empty). Use `derive` with op/path/value and
                   `true_value` / `false_value`.
- "constant"     : literal value (rare; use only when no DB path applies but
                   a stable constant should be stamped).
- "unmapped"     : no DB path plausibly supplies the value.

For radio_group / checkbox fields:
- Set `value.kind` to "constant" with literal=null.
- For each option in the field's `options` list, emit one OptionResolution
  whose `when` is a Resolution (scalar / derived / constant) that determines
  selection. Use `selected_if="equals"` with `equals_value` for derived
  resolutions, or `"truthy"` to select when the resolved value is non-empty.

Rules:
- NEVER invent paths not in the provided list.
- NEVER reference field labels, section names, or option values from any
  specific form in your reasoning. Work only from what you are given.
- Apply `transform` when the field_type clearly requires coercion:
  "date:<out_format>" (use the field's format_hint as out_format if given;
  default "YYYY-MM-DD"), "phone:formatted", "phone:digits", "number",
  "currency", "upper", "lower", "strip".
- Confidence: "high" for unambiguous matches, "medium" for plausible,
  "low" for guesses, "unmapped" if no plausible path exists.
- Set `reason` to one short sentence explaining the choice.

Return every input field with exactly one FieldResolution, in the same order.
"""

def _field_summary(f: FormField) -> dict:
    return dict(
        field_id=f.field_id, label=f.label, section=f.section,
        field_type=f.field_type, options=f.options,
        format_hint=f.format_hint, required=f.required,
    )

def _flatten_db(data: Any, prefix: str = "") -> dict[str, Any]:
    out: dict[str, Any] = {}
    if isinstance(data, dict):
        for k, v in data.items():
            key = f"{prefix}.{k}" if prefix else k
            out.update(_flatten_db(v, key))
    elif isinstance(data, list) and data:
        for i, item in enumerate(data):
            out.update(_flatten_db(item, f"{prefix}[{i}]"))
    else:
        out[prefix] = data
    return out

def _normalize_path(path: str) -> str:
    out, i = "", 0
    while i < len(path):
        if path[i] == "[":
            j = path.index("]", i); out += "[]"; i = j + 1
        else:
            out += path[i]; i += 1
    return out

def _sample_values(db: dict, n: int = 60) -> dict:
    flat = _flatten_db(db)
    seen: dict[str, Any] = {}
    for k, v in flat.items():
        if v in ("", None, [], {}): continue
        norm = _normalize_path(k)
        if norm in seen: continue
        seen[norm] = v
        if len(seen) >= n: break
    return seen

def llm_resolve(fields: list[FormField], paths: list[str], db: dict, model: str = "gpt-5-mini") -> list[FieldResolution]:
    if not fields: return []
    client = OpenAI(api_key=OPENAI_API_KEY)
    user_msg = json.dumps(dict(
        form_fields=[_field_summary(f) for f in fields],
        db_paths=paths,
        db_sample_values=_sample_values(db),
    ), indent=2)
    resp = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": LLM_SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        response_format=LLMResolverResponse,
    )
    parsed = resp.choices[0].message.parsed
    if parsed is None:
        raise RuntimeError(f"empty parse; finish_reason={resp.choices[0].finish_reason}")
    by_id = {fr.field_id: fr for fr in parsed.field_resolutions}
    return [
        by_id.get(f.field_id, FieldResolution(
            field_id=f.field_id, confidence="unmapped",
            value=Resolution(kind="unmapped", reason="missing from LLM response"),
        ))
        for f in fields
    ]

llm_hits = llm_resolve(residuals, ALL_PATHS, db)
print(f"LLM resolved: {sum(1 for fr in llm_hits if fr.confidence != 'unmapped')}/{len(llm_hits)}")
print()
for fr in llm_hits[:8]:
    p = fr.value.paths[0] if fr.value.paths else "-"
    print(f"  {fr.field_id:35s} kind={fr.value.kind:12s} conf={fr.confidence:8s} -> {p}")

## 8. Persist the `FieldMap` (form-keyed cache)

Merge heuristic + LLM resolutions, preserving the schema's field order, and write to `<stem>.fillmap.json`. On the next run with the same form + same DB schema fingerprint, both LLM calls are skipped.

In [ ]:
field_resolutions: list[FieldResolution] = []
llm_by_id = {fr.field_id: fr for fr in llm_hits}
for f in schema.fields:
    if f.field_id in heuristic_hits:
        field_resolutions.append(heuristic_hits[f.field_id])
    else:
        field_resolutions.append(llm_by_id[f.field_id])

fmap = FieldMap(
    schema_fingerprint=SCHEMA_FP,
    db_schema_fingerprint=DB_FP,
    form_pdf_basename=STEM,
    created_at=datetime.now(timezone.utc).isoformat(),
    field_resolutions=field_resolutions,
    unresolved_field_ids=[fr.field_id for fr in field_resolutions if fr.confidence == "unmapped"],
    notes=[
        f"heuristic resolved {len(heuristic_hits)}/{len(schema.fields)}",
        f"LLM resolved {sum(1 for fr in llm_hits if fr.confidence != 'unmapped')}/{len(llm_hits)}",
    ],
)

FILLMAP_JSON.write_text(fmap.model_dump_json(indent=2))
print(f"wrote {FILLMAP_JSON}")
by_kind: dict[str, int] = {}
for fr in field_resolutions:
    by_kind[fr.value.kind] = by_kind.get(fr.value.kind, 0) + 1
print(f"resolution kinds:  {by_kind}")
print(f"unresolved fields: {fmap.unresolved_field_ids}")

## 9. Materialize — `FieldMap` + DB → `FilledFormSchema`

Deterministic application. No LLM, no network. For every `FormField`:
1. Look up its `FieldResolution`.
2. Resolve the raw value(s) per `kind` (`scalar` / `compose` / `service_line` / `derived` / `constant`).
3. Apply the `transform` (date/phone/number/currency/upper/lower/strip).
4. For choice fields, evaluate each `OptionResolution.when` and accumulate selected options.

Unmapped or unresolvable fields produce `value=None` and surface in the review report.

In [ ]:
# --- value coercers (pure) ---
_DATE_PATTERNS = [
    ("%Y%m%d",   r"^\d{8}$"),
    ("%m/%d/%Y", r"^\d{1,2}/\d{1,2}/\d{4}$"),
    ("%m-%d-%Y", r"^\d{1,2}-\d{1,2}-\d{4}$"),
    ("%Y-%m-%d", r"^\d{4}-\d{1,2}-\d{1,2}$"),
]
_DATE_OUT = {
    "YYYY-MM-DD": "%Y-%m-%d", "MM/DD/YYYY": "%m/%d/%Y",
    "MM-DD-YYYY": "%m-%d-%Y", "YYYYMMDD":   "%Y%m%d",
    "DD/MM/YYYY": "%d/%m/%Y",
}

def coerce_date(raw: str, out_format: Optional[str] = None) -> Optional[str]:
    if not raw or not isinstance(raw, str): return None
    s = raw.strip()
    if not s: return None
    parsed = None
    for fmt, pat in _DATE_PATTERNS:
        if re.match(pat, s):
            try: parsed = datetime.strptime(s, fmt); break
            except ValueError: continue
    if parsed is None: return None
    return parsed.strftime(_DATE_OUT.get(out_format or "YYYY-MM-DD", "%Y-%m-%d"))

def coerce_phone(raw: str, style: str = "formatted") -> Optional[str]:
    if not raw or not isinstance(raw, str): return None
    digits = re.sub(r"\D", "", raw)
    if len(digits) < 7: return None
    if style == "digits": return digits
    if len(digits) == 10: return f"({digits[0:3]}) {digits[3:6]}-{digits[6:]}"
    if len(digits) == 11 and digits[0] == "1":
        return f"+1 ({digits[1:4]}) {digits[4:7]}-{digits[7:]}"
    return digits

def coerce_number(raw: str) -> Optional[str]:
    if raw is None or raw == "": return None
    s = str(raw).replace(",", "").strip()
    try: f = float(s)
    except ValueError: return None
    return str(int(f)) if f == int(f) else f"{f}"

def coerce_currency(raw: str) -> Optional[str]:
    if raw is None or raw == "": return None
    s = str(raw).replace("$", "").replace(",", "").strip()
    try: f = float(s)
    except ValueError: return None
    return f"{f:,.2f}"

def apply_transform(raw: Any, transform: Optional[str]) -> Optional[str]:
    if raw is None or raw == "": return None
    if transform is None: return str(raw)
    if transform.startswith("date:"):
        return coerce_date(str(raw), out_format=transform.split(":", 1)[1] or None)
    if transform == "phone:digits":            return coerce_phone(str(raw), style="digits")
    if transform in ("phone", "phone:formatted"): return coerce_phone(str(raw), style="formatted")
    if transform == "number":                  return coerce_number(str(raw))
    if transform == "currency":                return coerce_currency(str(raw))
    if transform == "upper":                   return str(raw).upper()
    if transform == "lower":                   return str(raw).lower()
    if transform == "strip":                   return str(raw).strip()
    raise ValueError(f"Unknown transform: {transform!r}")

def compose(values: list, template: str) -> Optional[str]:
    stringified = ["" if v is None else str(v) for v in values]
    rendered = re.sub(r"\s+", " ", template.format(*stringified)).strip()
    return rendered if rendered else None

# --- path lookup with [N] indexing ---
def db_path_get(data: dict, dotted: str) -> Any:
    parts: list = []; buf = ""; i = 0
    while i < len(dotted):
        ch = dotted[i]
        if ch == ".":
            if buf: parts.append(buf); buf = ""
            i += 1
        elif ch == "[":
            if buf: parts.append(buf); buf = ""
            j = dotted.index("]", i)
            parts.append(int(dotted[i+1:j])); i = j + 1
        else:
            buf += ch; i += 1
    if buf: parts.append(buf)
    cur = data
    for p in parts:
        if cur is None: return None
        if isinstance(p, int):
            if not isinstance(cur, list) or p >= len(cur): return None
            cur = cur[p]
        else:
            if not isinstance(cur, dict) or p not in cur: return None
            cur = cur[p]
    return cur

# --- resolution walker ---
def resolve_value(r: Resolution, db: dict) -> Any:
    if r.kind == "constant":    return r.literal
    if r.kind == "unmapped":    return None
    if r.kind == "scalar":      return db_path_get(db, r.paths[0]) if r.paths else None
    if r.kind == "compose":
        raws = [db_path_get(db, p) for p in r.paths]
        return compose(raws, r.template or "")
    if r.kind == "service_line":
        idx = r.service_line_index or 0
        if not r.paths: return None
        path = r.paths[0].replace("{i}", str(idx)).replace("[]", f"[{idx}]")
        return db_path_get(db, path)
    if r.kind == "derived":
        d = r.derive
        if d is None: return None
        observed = db_path_get(db, d.path)
        if d.op == "eq":         matched = str(observed) == str(d.value or "")
        elif d.op == "neq":      matched = str(observed) != str(d.value or "")
        elif d.op == "non_empty": matched = observed not in (None, "")
        elif d.op == "empty":    matched = observed in (None, "")
        else: raise ValueError(f"Unknown derived op: {d.op!r}")
        return d.true_value if matched else d.false_value
    raise ValueError(f"Unknown Resolution.kind: {r.kind!r}")

def option_matches(opt: OptionResolution, db: dict) -> bool:
    val = resolve_value(opt.when, db)
    if opt.selected_if == "always":  return val is not None and val != ""
    if opt.selected_if == "equals":  return str(val) == (opt.equals_value or "")
    return val is not None and val != "" and val is not False

def fill_from_db(schema: FormSchema, fmap: FieldMap, db: dict) -> FilledFormSchema:
    by_id = {fr.field_id: fr for fr in fmap.field_resolutions}
    filled: list[FilledFormField] = []
    for f in schema.fields:
        fr = by_id.get(f.field_id)
        if fr is None:
            filled.append(FilledFormField(**f.model_dump(), value=None)); continue
        if f.field_type in ("checkbox", "radio_group"):
            sel = [o.option_label for o in fr.options if option_matches(o, db)]
            v = (sel[0] if sel else None) if f.field_type == "radio_group" else (sel if sel else None)
        else:
            raw = resolve_value(fr.value, db)
            v = apply_transform(raw, fr.value.transform) if raw is not None else None
        filled.append(FilledFormField(**f.model_dump(), value=v))
    return FilledFormSchema.from_schema(schema, filled)

filled = fill_from_db(schema, fmap, db)
FILLED_JSON.write_text(filled.model_dump_json(indent=2))
print(f"wrote {FILLED_JSON}\n")
for f in filled.fields:
    v = f.value
    if isinstance(v, list): vs = repr(v)
    elif v is None:         vs = "(empty)"
    else:                   vs = str(v)[:70]
    print(f"  {f.field_id:55s} -> {vs}")

## 10. Write the filled PDF

Hand the `FilledFormSchema` and `AcroFormBinding` to `write_acroform` (`claims_parser.acroform_writer`). For each field, it locates the widget by xref, sets `widget.field_value`, calls `widget.update()`, and records any field it couldn't stamp in the review report. Pass `flatten=True` to bake widgets into static content; default is editable output.

The writer is imported rather than inlined — it's mature widget-update code that's not the educational focus here.

In [ ]:
review = write_acroform(
    pdf_path=INPUT_PDF,
    filled=filled,
    binding=binding,
    output_path=FILLED_PDF,
    flatten=False,
)
REVIEW_JSON.write_text(review.model_dump_json(indent=2))
print(f"wrote {FILLED_PDF}")
print(f"wrote {REVIEW_JSON}")

counts = review.by_reason()
if counts:
    print("\nReview flags by reason:")
    for k, v in counts.items():
        print(f"  {k:30s} {v}")
else:
    print("\nNo fields flagged for review.")

## 11. Summary

What was produced, where it lives, and what runs next time the same form is filled for a different case.

In [ ]:
print("Per-run artifacts:")
for p in [SCHEMA_JSON, BINDING_JSON, FILLMAP_JSON, FILLED_JSON, REVIEW_JSON]:
    print(f"  {p}  ({p.stat().st_size / 1024:.1f} KB)")
print(f"  {FILLED_PDF}  ({FILLED_PDF.stat().st_size / 1024:.1f} KB)")

total = len(filled.fields)
filled_n = sum(1 for f in filled.fields if f.value not in (None, "", []))
unmapped = len(fmap.unresolved_field_ids)
print(f"\nFields:  total={total}  filled={filled_n}  unmapped={unmapped}")

print("\nCache reuse:")
print(f"  schema_fingerprint    = {SCHEMA_FP}")
print(f"  db_schema_fingerprint = {DB_FP}")
print(f"  next run on the same form + DB schema -> 0 LLM calls (loads {FILLMAP_JSON.name})")
print(f"  next run on a different form -> 1 LLM call to build a new fillmap")
print(f"  next run on a new case (same form) -> 0 LLM calls; just re-materialize")